In [147]:
#讀取圖片上的點
import cv2

# 儲存點
points = []

# 標籤名稱
labels = [
    "Left Eye", "Right Eye", "Nose Bridge(for glasses)",
    "Left Mouth Corner", "Right Mouth Corner", "Philtrum (for beard)"
]

def click_event(event, x, y, flags, param):
    global points, img_display

    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 6:
            points.append((x, y))

            # 畫點
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)

            # 標文字
            label = labels[len(points)-1]
            cv2.putText(img_display, label, (x+10, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            cv2.imshow("image", img_display)

            print(f"{label}: ({x}, {y})")
        else:
            print("⚠️ 已經點滿 6 個點了（按 r 重設）")

# 讀圖片
img = cv2.imread(r"C:\opencv_project\face3.png")

# 縮放避免點偏移
scale = 0.7
img_small = cv2.resize(img, None, fx=scale, fy=scale)
img_display = img_small.copy()

cv2.namedWindow("image")
cv2.setMouseCallback("image", click_event)

print("👉 點選順序：左眼 → 右眼 → 鼻樑眉心 → 左嘴角 → 右嘴角 → 人中(鬍子)")
print("👉 按 r 重設，按 q 離開")

while True:
    cv2.imshow("image", img_display)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('r'):
        points = []
        img_display = img_small.copy()
        print("🔄 已重設")
    elif key == ord('q'):
        break

cv2.destroyAllWindows()

# 檢查點數:若沒滿足，當成未完成，值不加入變數
if len(points) == 6:
    # 眼鏡點
    left_eye, right_eye, nose_bridge = points[:3]
    left_eye = [int(left_eye[0]/scale), int(left_eye[1]/scale)]
    right_eye = [int(right_eye[0]/scale), int(right_eye[1]/scale)]
    nose_bridge = [int(nose_bridge[0]/scale), int(nose_bridge[1]/scale)]
    pts_glass_dst = [left_eye, right_eye, nose_bridge]

    # 鬍子點
    left_mouth, right_mouth, philtrum_beard = points[3:]
    left_mouth = [int(left_mouth[0]/scale), int(left_mouth[1]/scale)]
    right_mouth = [int(right_mouth[0]/scale), int(right_mouth[1]/scale)]
    philtrum_beard = [int(philtrum_beard[0]/scale), int(philtrum_beard[1]/scale)]
    pts_beard_dst = [left_mouth, right_mouth, philtrum_beard]

    print("\n✅ 眼鏡座標 (pts_glass_dst):", pts_glass_dst)
    print("✅ 鬍子座標 (pts_beard_dst):", pts_beard_dst)
else:
    print("\n⚠️ 點不足 6 個點")

👉 點選順序：左眼 → 右眼 → 鼻樑眉心 → 左嘴角 → 右嘴角 → 人中(鬍子)
👉 按 r 重設，按 q 離開
Left Eye: (160, 400)
Right Eye: (350, 344)
Nose Bridge(for glasses): (248, 330)
Left Mouth Corner: (229, 584)
Right Mouth Corner: (385, 536)
Philtrum (for beard): (306, 516)

✅ 眼鏡座標 (pts_glass_dst): [[228, 571], [500, 491], [354, 471]]
✅ 鬍子座標 (pts_beard_dst): [[327, 834], [550, 765], [437, 737]]


In [148]:
#矩陣變形程式及影像融合
import cv2
import numpy as np

# 1. 讀取影像與素材
img = cv2.imread(r"C:\opencv_project\face3.png") #臉的檔名
glass = cv2.imread(r"C:\opencv_project\glass.png", cv2.IMREAD_UNCHANGED)  # 必須讀取 Alpha 通道
beard = cv2.imread(r"C:\opencv_project\beard.png", cv2.IMREAD_UNCHANGED)

def apply_affine_prop(face_img, prop_img, face_pts, prop_pts):
    rows, cols, _ = face_img.shape
    
    # 計算仿射矩陣 (作業核心：從素材座標轉到人臉座標
    M = cv2.getAffineTransform(np.float32(prop_pts), np.float32(face_pts))
    
    # 執行變形
    warped_prop = cv2.warpAffine(prop_img, M, (cols, rows))
    
    # 分離顏色與透明度遮罩 (處理 Alpha Channel)
    if warped_prop.shape[2] == 4:
        b, g, r, a = cv2.split(warped_prop)
        overlay_color = cv2.merge((b, g, r))
        mask = a / 255.0
    else:
        overlay_color = warped_prop
        mask = np.ones((rows, cols)) # 若無透明層則全選

    # 進行合成 
    for c in range(3):
        face_img[:, :, c] = (1.0 - mask) * face_img[:, :, c] + mask * overlay_color[:, :, c]
    
    return face_img

# --- 定義特徵點 (Strategy) [cite: 21, 41] ---

# 2. 處理眼鏡 (對齊雙眼)
# 素材點 (左鏡片中心, 右鏡片中心, 鼻樑開始處)
h_g, w_g = glass.shape[:2]
pts_glass_src = [ [w_g * 0.28, h_g * 0.5], # 左鏡片中心
    [w_g * 0.72, h_g * 0.5], # 右鏡片中心
    [w_g * 0.5,  h_g * 0.25]]  # 鏡框上緣中心 (對應眉心位置) 

#照片上點 
pts_glass_dst
img = apply_affine_prop(img, glass, pts_glass_dst, pts_glass_src)

# 3. 處理鬍子 (對齊嘴角)
# 素材點 (左鬍鬚端, 右鬍鬚端, 鬍子中心)
h_b, w_b = beard.shape[:2]
pts_beard_src = [[w_b * 0.2, h_b * 0.95], 
                 [w_b * 0.8, h_b * 0.95], 
                 [w_b * 0.5, h_b * 0.3]]
# 照片點 (對應照片中的左右嘴角上方與人中)
pts_beard_dst 


img = apply_affine_prop(img, beard, pts_beard_dst, pts_beard_src)

display_scale = 0.7

# 使用 cv2.resize 縮小圖片
# fx, fy 為水平與垂直縮放比例；interpolation 用 INTER_AREA 效果最好
img_small = cv2.resize(img, None, fx=display_scale, fy=display_scale, interpolation=cv2.INTER_AREA)

# 展示結果 [cite: 48, 55]
cv2.imshow("Result", img_small)
cv2.imwrite("result_output.jpg", img_small) # 存檔用於簡報
cv2.waitKey(0)
cv2.destroyAllWindows()